In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("techsash/waste-classification-data")

print("Path to dataset files:", path)

/Users/enesdemir/Desktop/RecyclableProject/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: /Users/enesdemir/.cache/kagglehub/datasets/techsash/waste-classification-data/versions/1


In [2]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.losses import MeanSquaredError
from tensorflow.keras.layers import Dense
from tensorflow.keras.activations import relu,softmax,sigmoid
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import RootMeanSquaredError
from tensorflow.keras import metrics
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler,RobustScaler
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.models import Sequential,Model,load_model
from tensorflow.keras.layers import BatchNormalization, InputLayer,Dropout,Input,Conv2D,MaxPooling2D,Flatten

In [3]:
import kagglehub

path = kagglehub.dataset_download("alistairking/recyclable-and-household-waste-classification")

print("Path to dataset files:", path)

Path to dataset files: /Users/enesdemir/.cache/kagglehub/datasets/alistairking/recyclable-and-household-waste-classification/versions/1


In [4]:
import warnings
warnings.filterwarnings('ignore')

dataset_path = '/Users/enesdemir/Desktop/RecyclableProject/TrashType_Image_Dataset'

garbage_types = os.listdir(dataset_path)

print(garbage_types)

['paper', 'metal', 'cardboard', 'trash', 'glass', 'plastic']


In [5]:
from PIL import Image
dimensions = set()

for types in garbage_types:
    folder_path = os.path.join(dataset_path,types)

    if os.path.isdir(folder_path):
        images = [f for f in os.listdir(folder_path)  if f.endswith(('jpeg','jpg','png'))]

        image_len = len(images)
        print(f"{garbage_types} object contains {image_len} images.")

        for image_file in images:

            image_path = os.path.join(folder_path, image_file)
            with Image.open(image_path) as img:

                widht,height = img.size
                channels = len(img.getbands())
                dimensions.add((widht,height,channels))
        print(f"{garbage_types} images dimensions are {dimensions.pop()}")

['paper', 'metal', 'cardboard', 'trash', 'glass', 'plastic'] object contains 594 images.
['paper', 'metal', 'cardboard', 'trash', 'glass', 'plastic'] images dimensions are (512, 384, 3)
['paper', 'metal', 'cardboard', 'trash', 'glass', 'plastic'] object contains 410 images.
['paper', 'metal', 'cardboard', 'trash', 'glass', 'plastic'] images dimensions are (512, 384, 3)
['paper', 'metal', 'cardboard', 'trash', 'glass', 'plastic'] object contains 403 images.
['paper', 'metal', 'cardboard', 'trash', 'glass', 'plastic'] images dimensions are (512, 384, 3)
['paper', 'metal', 'cardboard', 'trash', 'glass', 'plastic'] object contains 137 images.
['paper', 'metal', 'cardboard', 'trash', 'glass', 'plastic'] images dimensions are (512, 384, 3)
['paper', 'metal', 'cardboard', 'trash', 'glass', 'plastic'] object contains 501 images.
['paper', 'metal', 'cardboard', 'trash', 'glass', 'plastic'] images dimensions are (512, 384, 3)
['paper', 'metal', 'cardboard', 'trash', 'glass', 'plastic'] object co

In [6]:
data =[]

for types in garbage_types:
    folder_path = os.path.join(dataset_path,types)
    for images in os.listdir(folder_path):
        data.append((os.path.join(folder_path,images),types))

df = pd.DataFrame(data)
df.columns = ['filepath','label']

In [7]:
df.head()

,filepath,label
0,/Users/enesdemir/Desktop/RecyclableProject/Tra...,paper
1,/Users/enesdemir/Desktop/RecyclableProject/Tra...,paper
2,/Users/enesdemir/Desktop/RecyclableProject/Tra...,paper
3,/Users/enesdemir/Desktop/RecyclableProject/Tra...,paper
4,/Users/enesdemir/Desktop/RecyclableProject/Tra...,paper


In [8]:
from sklearn.model_selection import train_test_split

train_df,test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])

train_df,validation_df = train_test_split(train_df, test_size=0.2, random_state=42, stratify=train_df['label'])


In [9]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_data_gen = ImageDataGenerator(
	rescale=1./255,
	rotation_range=45,
	width_shift_range=0.15,
	height_shift_range=0.15,
	shear_range=0.05,
	zoom_range=0.15,
	horizontal_flip=True,
    vertical_flip=True,
    brightness_range=[0.9,1.1],
    channel_shift_range=10,
	fill_mode='nearest'
)

validation_data_gen = ImageDataGenerator(rescale=1./255)

test_data_gen = ImageDataGenerator(rescale=1./255)

In [10]:
train_generator = train_data_gen.flow_from_dataframe(
    dataframe=train_df,
    x_col='filepath',
    y_col='label',
    target_size=(384,384),
    batch_size=32,
    class_mode='categorical',
    seed=42,
    shuffle=False
)

val_generator = validation_data_gen.flow_from_dataframe(
    dataframe=validation_df,                    
    x_col="filepath",                    
    y_col="label",                    
    target_size=(384, 384),        
    batch_size=32,                   
    class_mode='categorical',            
    seed=42,                            
    shuffle=False                       
)

test_generator = validation_data_gen.flow_from_dataframe(
    dataframe=test_df,                    
    x_col="filepath",                    
    y_col="label",                    
    target_size=(384, 384),        
    batch_size=32,                   
    class_mode='categorical',            
    seed=42,                            
    shuffle=False      
)
         

Found 1616 validated image filenames belonging to 6 classes.
Found 405 validated image filenames belonging to 6 classes.
Found 506 validated image filenames belonging to 6 classes.


In [11]:
print(f"Number of batches in train_generator: {len(train_generator)}")
print(f"Number of batches in val_generator: {len(val_generator)}")
print(f"Number of batches in val_generator: {len(test_generator)}")

Number of batches in train_generator: 51
Number of batches in val_generator: 13
Number of batches in val_generator: 16


In [12]:
model = Sequential()

model.add(Conv2D(32,(3,3),activation='relu',name='layer1',input_shape=(384,384,3)))
model.add(MaxPooling2D(pool_size=(2,2)))

model.add(Conv2D(64,(3,3),activation='relu',name='layer2'))
model.add(MaxPooling2D(pool_size=(2,2)))

model.add(Conv2D(128,(3,3),activation='relu',name='layer3'))
model.add(MaxPooling2D(pool_size=(2,2)))

model.add(Flatten())
model.add(Dense(512,activation='relu',name='layer4'))
model.add(Dropout(0.4))
model.add(Dense(6,activation='softmax',name='output_layer'))


In [13]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ layer1 (Conv2D)                 │ (None, 382, 382, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 191, 191, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer2 (Conv2D)                 │ (None, 189, 189, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 94, 94, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer3 (Conv2D)                 │ (None, 92, 92, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 46, 46, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 270848)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer4 (Dense)                  │ (None, 512)            │   138,674,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_layer (Dense)            │ (None, 6)              │         3,078 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 138,771,014 (529.37 MB)

 Trainable params: 138,771,014 (529.37 MB)

 Non-trainable params: 0 (0.00 B)

In [14]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [15]:
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

In [16]:
history1 = model.fit(
    train_generator,
    steps_per_epoch=len(train_generator),
    validation_data=val_generator,
    validation_steps=len(val_generator),
    epochs=30,
    callbacks=[early_stopping]
)

Epoch 1/30
51/51 ━━━━━━━━━━━━━━━━━━━━ 74s 1s/step - accuracy: 0.2731 - loss: 3.7698 - val_accuracy: 0.3975 - val_loss: 1.4237
Epoch 2/30
51/51 ━━━━━━━━━━━━━━━━━━━━ 71s 1s/step - accuracy: 0.4036 - loss: 1.4294 - val_accuracy: 0.4691 - val_loss: 1.3002
Epoch 3/30
51/51 ━━━━━━━━━━━━━━━━━━━━ 71s 1s/step - accuracy: 0.4627 - loss: 1.3495 - val_accuracy: 0.4321 - val_loss: 1.3502
Epoch 4/30
51/51 ━━━━━━━━━━━━━━━━━━━━ 75s 1s/step - accuracy: 0.4592 - loss: 1.3395 - val_accuracy: 0.5086 - val_loss: 1.2111
Epoch 5/30
51/51 ━━━━━━━━━━━━━━━━━━━━ 118s 2s/step - accuracy: 0.5306 - loss: 1.2176 - val_accuracy: 0.5333 - val_loss: 1.1634
Epoch 6/30
51/51 ━━━━━━━━━━━━━━━━━━━━ 86s 2s/step - accuracy: 0.4997 - loss: 1.2553 - val_accuracy: 0.4370 - val_loss: 1.3616
Epoch 7/30
51/51 ━━━━━━━━━━━━━━━━━━━━ 88s 2s/step - accuracy: 0.4987 - loss: 1.2469 - val_accuracy: 0.5358 - val_loss: 1.1909
Epoch 8/30
51/51 ━━━━━━━━━━━━━━━━━━━━ 96s 2s/step - accuracy: 0.5402 - loss: 1.2015 - val_accuracy: 0.5432 - val_loss

In [17]:
model2 = Sequential()

model2.add(Conv2D(32,(3,3),activation='relu',name='layer1',input_shape=(384,384,3)))
model2.add(MaxPooling2D(pool_size=(2,2)))

model2.add(Conv2D(64,(3,3),activation='relu',name='layer2'))
model2.add(MaxPooling2D(pool_size=(2,2)))

model2.add(Conv2D(128,(3,3),activation='relu',name='layer3'))
model2.add(MaxPooling2D(pool_size=(2,2)))

model2.add(Flatten())
model2.add(Dense(512,activation='relu',name='layer4'))
model2.add(Dropout(0.5))
model2.add(Dense(6,activation='softmax',name='output_layer'))


In [18]:
model2.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ layer1 (Conv2D)                 │ (None, 382, 382, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 191, 191, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer2 (Conv2D)                 │ (None, 189, 189, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 94, 94, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer3 (Conv2D)                 │ (None, 92, 92, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 46, 46, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 270848)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer4 (Dense)                  │ (None, 512)            │   138,674,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_layer (Dense)            │ (None, 6)              │         3,078 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 138,771,014 (529.37 MB)

 Trainable params: 138,771,014 (529.37 MB)

 Non-trainable params: 0 (0.00 B)

In [19]:
model2.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history2 = model2.fit(
    train_generator,
    steps_per_epoch=len(train_generator),
    validation_data=val_generator,
    validation_steps=len(val_generator),
    epochs=30,
    callbacks=[early_stopping]
)

Epoch 1/30
51/51 ━━━━━━━━━━━━━━━━━━━━ 93s 2s/step - accuracy: 0.2421 - loss: 3.9021 - val_accuracy: 0.3778 - val_loss: 1.4258
Epoch 2/30
51/51 ━━━━━━━━━━━━━━━━━━━━ 96s 2s/step - accuracy: 0.3850 - loss: 1.4829 - val_accuracy: 0.4173 - val_loss: 1.3783
Epoch 3/30
51/51 ━━━━━━━━━━━━━━━━━━━━ 94s 2s/step - accuracy: 0.3878 - loss: 1.4486 - val_accuracy: 0.4247 - val_loss: 1.4316
Epoch 4/30
51/51 ━━━━━━━━━━━━━━━━━━━━ 96s 2s/step - accuracy: 0.4233 - loss: 1.3600 - val_accuracy: 0.4198 - val_loss: 1.3973
Epoch 5/30
51/51 ━━━━━━━━━━━━━━━━━━━━ 95s 2s/step - accuracy: 0.4458 - loss: 1.3780 - val_accuracy: 0.4667 - val_loss: 1.3387
Epoch 6/30
51/51 ━━━━━━━━━━━━━━━━━━━━ 100s 2s/step - accuracy: 0.4492 - loss: 1.3674 - val_accuracy: 0.4790 - val_loss: 1.3684
Epoch 7/30
51/51 ━━━━━━━━━━━━━━━━━━━━ 100s 2s/step - accuracy: 0.4848 - loss: 1.2407 - val_accuracy: 0.4543 - val_loss: 1.2414
Epoch 8/30
51/51 ━━━━━━━━━━━━━━━━━━━━ 101s 2s/step - accuracy: 0.4828 - loss: 1.2763 - val_accuracy: 0.5012 - val_lo

In [20]:
model3 = Sequential()

model3.add(Conv2D(32,(3,3),activation='relu',name='layer1',input_shape=(384,384,3)))
model3.add(MaxPooling2D(pool_size=(2,2)))

model3.add(Conv2D(64,(3,3),activation='relu',name='layer2'))
model3.add(MaxPooling2D(pool_size=(2,2)))

model3.add(Conv2D(128,(3,3),activation='relu',name='layer3'))
model3.add(MaxPooling2D(pool_size=(2,2)))

model3.add(Flatten())
model3.add(Dense(256,activation='relu',name='layer4'))
model3.add(Dropout(0.5))
model3.add(Dense(6,activation='softmax',name='output_layer'))


In [21]:
model3.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ layer1 (Conv2D)                 │ (None, 382, 382, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 191, 191, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer2 (Conv2D)                 │ (None, 189, 189, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 94, 94, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer3 (Conv2D)                 │ (None, 92, 92, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_8 (MaxPooling2D)  │ (None, 46, 46, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 270848)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer4 (Dense)                  │ (None, 256)            │    69,337,344 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_layer (Dense)            │ (None, 6)              │         1,542 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 69,432,134 (264.86 MB)

 Trainable params: 69,432,134 (264.86 MB)

 Non-trainable params: 0 (0.00 B)

In [22]:
model3.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history3 = model3.fit(
    train_generator,
    steps_per_epoch=len(train_generator),
    validation_data=val_generator,
    validation_steps=len(val_generator),
    epochs=30,
    callbacks=[early_stopping]
)

Epoch 1/30
51/51 ━━━━━━━━━━━━━━━━━━━━ 84s 2s/step - accuracy: 0.2190 - loss: 3.0054 - val_accuracy: 0.3160 - val_loss: 1.5517
Epoch 2/30
51/51 ━━━━━━━━━━━━━━━━━━━━ 81s 2s/step - accuracy: 0.3677 - loss: 1.5168 - val_accuracy: 0.4543 - val_loss: 1.3693
Epoch 3/30
51/51 ━━━━━━━━━━━━━━━━━━━━ 73s 1s/step - accuracy: 0.3952 - loss: 1.4183 - val_accuracy: 0.4494 - val_loss: 1.3370
Epoch 4/30
51/51 ━━━━━━━━━━━━━━━━━━━━ 83s 2s/step - accuracy: 0.3991 - loss: 1.4028 - val_accuracy: 0.4346 - val_loss: 1.4180
Epoch 5/30
51/51 ━━━━━━━━━━━━━━━━━━━━ 75s 1s/step - accuracy: 0.4238 - loss: 1.4099 - val_accuracy: 0.4716 - val_loss: 1.3101
Epoch 6/30
51/51 ━━━━━━━━━━━━━━━━━━━━ 80s 2s/step - accuracy: 0.4669 - loss: 1.3780 - val_accuracy: 0.4346 - val_loss: 1.3673
Epoch 7/30
51/51 ━━━━━━━━━━━━━━━━━━━━ 68s 1s/step - accuracy: 0.4558 - loss: 1.3594 - val_accuracy: 0.4346 - val_loss: 1.3210
Epoch 8/30
51/51 ━━━━━━━━━━━━━━━━━━━━ 68s 1s/step - accuracy: 0.4792 - loss: 1.2791 - val_accuracy: 0.5037 - val_loss:

In [23]:
model4 = Sequential()

model4.add(Conv2D(32,(3,3),activation='relu',name='layer1',input_shape=(384,384,3)))
model4.add(MaxPooling2D(pool_size=(2,2)))

model4.add(Conv2D(64,(3,3),activation='relu',name='layer2'))
model4.add(MaxPooling2D(pool_size=(2,2)))

model4.add(Conv2D(128,(3,3),activation='relu',name='layer3'))
model4.add(MaxPooling2D(pool_size=(2,2)))

model4.add(Conv2D(256,(3,3),activation='relu',name='layer3last'))
model4.add(MaxPooling2D(pool_size=(2,2)))

model4.add(Flatten())
model4.add(Dense(512,activation='relu',name='layer4'))
model4.add(Dropout(0.5))
model4.add(Dense(6,activation='softmax',name='output_layer'))


In [24]:
model4.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ layer1 (Conv2D)                 │ (None, 382, 382, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_9 (MaxPooling2D)  │ (None, 191, 191, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer2 (Conv2D)                 │ (None, 189, 189, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_10 (MaxPooling2D) │ (None, 94, 94, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer3 (Conv2D)                 │ (None, 92, 92, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_11 (MaxPooling2D) │ (None, 46, 46, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer3last (Conv2D)             │ (None, 44, 44, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_12 (MaxPooling2D) │ (None, 22, 22, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_3 (Flatten)             │ (None, 123904)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer4 (Dense)                  │ (None, 512)            │    63,439,360 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_layer (Dense)            │ (None, 6)              │         3,078 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 63,830,854 (243.50 MB)

 Trainable params: 63,830,854 (243.50 MB)

 Non-trainable params: 0 (0.00 B)

In [25]:
model4.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history4 = model4.fit(
    train_generator,
    steps_per_epoch=len(train_generator),
    validation_data=val_generator,
    validation_steps=len(val_generator),
    epochs=30,
    callbacks=[early_stopping]
)

Epoch 1/30
51/51 ━━━━━━━━━━━━━━━━━━━━ 82s 2s/step - accuracy: 0.2243 - loss: 2.1885 - val_accuracy: 0.3457 - val_loss: 1.5190
Epoch 2/30
51/51 ━━━━━━━━━━━━━━━━━━━━ 83s 2s/step - accuracy: 0.3387 - loss: 1.5608 - val_accuracy: 0.3728 - val_loss: 1.4831
Epoch 3/30
51/51 ━━━━━━━━━━━━━━━━━━━━ 81s 2s/step - accuracy: 0.4131 - loss: 1.4138 - val_accuracy: 0.4296 - val_loss: 1.3819
Epoch 4/30
51/51 ━━━━━━━━━━━━━━━━━━━━ 83s 2s/step - accuracy: 0.4053 - loss: 1.4156 - val_accuracy: 0.4716 - val_loss: 1.3288
Epoch 5/30
51/51 ━━━━━━━━━━━━━━━━━━━━ 84s 2s/step - accuracy: 0.4622 - loss: 1.3138 - val_accuracy: 0.4790 - val_loss: 1.3636
Epoch 6/30
51/51 ━━━━━━━━━━━━━━━━━━━━ 82s 2s/step - accuracy: 0.4620 - loss: 1.3236 - val_accuracy: 0.4346 - val_loss: 1.4729
Epoch 7/30
51/51 ━━━━━━━━━━━━━━━━━━━━ 83s 2s/step - accuracy: 0.4757 - loss: 1.3061 - val_accuracy: 0.5136 - val_loss: 1.2945
Epoch 8/30
51/51 ━━━━━━━━━━━━━━━━━━━━ 83s 2s/step - accuracy: 0.4896 - loss: 1.2720 - val_accuracy: 0.4543 - val_loss:

In [26]:
model5 = Sequential()

model5.add(Conv2D(32,(3,3),activation='relu',name='layer1',input_shape=(384,384,3)))
model5.add(MaxPooling2D(pool_size=(2,2)))

model5.add(Conv2D(64,(3,3),activation='relu',name='layer2'))
model5.add(MaxPooling2D(pool_size=(2,2)))

model5.add(Conv2D(128,(3,3),activation='relu',name='layer3'))
model5.add(MaxPooling2D(pool_size=(2,2)))

model5.add(Conv2D(256,(3,3),activation='relu',name='layer4'))
model5.add(MaxPooling2D(pool_size=(2,2)))

model5.add(Flatten())
model5.add(Dense(256,activation='relu',name='layer5'))
BatchNormalization()

model5.add(Dense(512,activation='relu',name='layer6'))

model5.add(Dropout(0.5))
model5.add(Dense(6,activation='softmax',name='output_layer'))


In [27]:
model5.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ layer1 (Conv2D)                 │ (None, 382, 382, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_13 (MaxPooling2D) │ (None, 191, 191, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer2 (Conv2D)                 │ (None, 189, 189, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_14 (MaxPooling2D) │ (None, 94, 94, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer3 (Conv2D)                 │ (None, 92, 92, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_15 (MaxPooling2D) │ (None, 46, 46, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer4 (Conv2D)                 │ (None, 44, 44, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_16 (MaxPooling2D) │ (None, 22, 22, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_4 (Flatten)             │ (None, 123904)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer5 (Dense)                  │ (None, 256)            │    31,719,680 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer6 (Dense)                  │ (None, 512)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_layer (Dense)            │ (None, 6)              │         3,078 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 32,242,758 (123.00 MB)

 Trainable params: 32,242,758 (123.00 MB)

 Non-trainable params: 0 (0.00 B)

In [28]:
model5.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history5 = model5.fit(
    train_generator,
    steps_per_epoch=len(train_generator),
    validation_data=val_generator,
    validation_steps=len(val_generator),
    epochs=30,
    callbacks=[early_stopping]
)

Epoch 1/30
51/51 ━━━━━━━━━━━━━━━━━━━━ 84s 2s/step - accuracy: 0.2607 - loss: 2.2606 - val_accuracy: 0.3333 - val_loss: 1.5939
Epoch 2/30
51/51 ━━━━━━━━━━━━━━━━━━━━ 79s 2s/step - accuracy: 0.3257 - loss: 1.5477 - val_accuracy: 0.3383 - val_loss: 1.5012
Epoch 3/30
51/51 ━━━━━━━━━━━━━━━━━━━━ 79s 2s/step - accuracy: 0.3528 - loss: 1.5028 - val_accuracy: 0.3926 - val_loss: 1.4758
Epoch 4/30
51/51 ━━━━━━━━━━━━━━━━━━━━ 76s 1s/step - accuracy: 0.4165 - loss: 1.4227 - val_accuracy: 0.4494 - val_loss: 1.3422
Epoch 5/30
51/51 ━━━━━━━━━━━━━━━━━━━━ 76s 1s/step - accuracy: 0.4691 - loss: 1.3306 - val_accuracy: 0.4988 - val_loss: 1.2510
Epoch 6/30
51/51 ━━━━━━━━━━━━━━━━━━━━ 77s 1s/step - accuracy: 0.4843 - loss: 1.2663 - val_accuracy: 0.5481 - val_loss: 1.2034
Epoch 7/30
51/51 ━━━━━━━━━━━━━━━━━━━━ 81s 2s/step - accuracy: 0.5209 - loss: 1.2231 - val_accuracy: 0.5605 - val_loss: 1.1155
Epoch 8/30
51/51 ━━━━━━━━━━━━━━━━━━━━ 82s 2s/step - accuracy: 0.5337 - loss: 1.1890 - val_accuracy: 0.5111 - val_loss: